# TIF Econometría II: Shocks de El Niño Costero y Mercado Bursátil Peruano
## Modelo de Series de Tiempo: BVL General, ICEN y Expectativas (2004 - 2026)

### Diccionario Breve de Variables y Códigos Oficiales

| Variable | Código | Fuente | Rol en el Modelo | Descripción Corta | Unidad / Transformación |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **`date_str`** | — | Calendario | Tiempo | Fecha mensual (YYYY-MM) | Texto |
| **`year`** / **`month`** | — | Calendario | Tiempo | Año y mes cronológico | Entero |
| **`stata_tm`** | — | Calendario | Tiempo | Fecha en formato Stata (`%tm`) | Meses desde 1960m1 |
| **`bvl_general`** | `PN01142MM` | BCRP | **Dependiente** | Índice General de la BVL (S&P/BVL) | Puntos (cierre mensual) |
| **`ret_bvl_general`** | — | BCRP | **Dependiente** | Retorno continuo de la BVL General | $100 \times \ln(P_t / P_{t-1})$ (%) |
| **`icen`** | — | IGP / ENFEN | **Explicativa** | Índice Costero El Niño (Niño 1+2) | Anomalía térmica TSM (°C) |
| **`icen_calido`** | — | IGP / ENFEN | Explicativa | Shock cálido de El Niño | $\max(ICEN_t, 0)$ (°C) |
| **`icen_frio`** | — | IGP / ENFEN | Explicativa | Fase fría de La Niña | $\min(ICEN_t, 0)$ (°C) |
| **`dummy_nino`** | — | IGP / ENFEN | Explicativa | Evento Niño moderado/fuerte | 1 si $ICEN_t \ge 1.0$, 0 caso contrario |
| **`exp_inflacion_12m`**| `PD12912AM` | BCRP | **Expectativa** | Expectativa de Inflación a 12 meses | % anual (encuesta BCRP) |
| **`d_exp_inflacion_12m`**| — | BCRP | Expectativa | Cambio en expectativa de inflación | Primera diferencia ($\Delta$ p.p.) |
| **`exp_emp_economia_3m`**| `PD38045AM` | BCRP | **Expectativa** | Confianza Empresarial Economía a 3m| Índice de difusión (>50 optimismo) |
| **`d_exp_emp_economia_3m`**| — | BCRP | Expectativa | Cambio en la confianza empresarial | Primera diferencia ($\Delta$ puntos) |
| **`tc_pen_usd`** | `PN01234PM` | BCRP | Control Macro | Tipo de cambio promedio interbancario | S/ por USD |
| **`ret_tc_pen_usd`** | — | BCRP | Control Macro | Tasa de depreciación cambiaria | $100 \times \ln(TC_t / TC_{t-1})$ (%) |
| **`tasa_ref`** | `PD04722MM` | BCRP | Control Macro | Tasa de referencia de política monetaria | % anual |
| **`d_tasa_ref`** | — | BCRP | Control Macro | Variación de la tasa de política monetaria | Primera diferencia ($\Delta$ p.p.) |
| **`cobre_lme`** | `PN01652XM` | BCRP | Control Macro | Cotización internacional Cobre LME | Centavos de US$ por libra |
| **`ret_cobre_lme`** | — | BCRP | Control Macro | Retorno del precio del cobre | $100 \times \ln(Cu_t / Cu_{t-1})$ (%) |
| **`petroleo_wti`** | `PN01660XM` | BCRP | Control Macro | Cotización Petróleo WTI | US$ por barril |
| **`ret_petroleo_wti`** | — | BCRP | Control Macro | Retorno del precio del petróleo | $100 \times \ln(WTI_t / WTI_{t-1})$ (%) |
| **`pbi_indice`** | `PN01770AM` | BCRP/INEI | Control Macro | Índice de Actividad Económica (PBI)| Índice 2007 = 100 |
| **`ret_pbi_indice`** | — | BCRP/INEI | Control Macro | Crecimiento mensual del PBI | $100 \times \ln(PBI_t / PBI_{t-1})$ (%) |
| **`sp500`** | `^GSPC` | Yahoo Fin. | Control Global | Índice S&P 500 (mercado accionario) | Puntos (cierre mensual) |
| **`ret_sp500`** | — | Yahoo Fin. | Control Global | Retorno bursátil global de referencia | $100 \times \ln(SP_t / SP_{t-1})$ (%) |
| **`vix`** | `^VIX` | Yahoo Fin. | Control Global | Índice de Volatilidad CBOE (VIX) | Puntos (aversión al riesgo global) |
| **`dummy_crisis2008`** | — | Externa | Quiebre | Crisis Financiera Subprime | 1 si 2008m9 a 2009m6, 0 otros |
| **`dummy_covid`** | — | Externa | Quiebre | Shock pandémico COVID-19 | 1 si 2020m3 a 2020m12, 0 otros |

In [ ]:
import io
import json
import requests
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

# Configuración de entorno y estilo visual
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Librerías importadas exitosamente.")

### 1. Extracción de Series desde la API del BCRP (2004 - 2026)

In [ ]:
# Códigos oficiales BCRP
BCRP_SERIES = {
    'bvl_general': 'PN01142MM',        # S&P/BVL Peru General
    'exp_inflacion_12m': 'PD12912AM',  # Expectativa de Inflación a 12 meses (%)
    'exp_emp_economia_3m': 'PD38045AM',# Confianza Empresarial Economía a 3 meses
    'tc_pen_usd': 'PN01234PM',         # Tipo de cambio promedio interbancario (S/ por USD)
    'tasa_ref': 'PD04722MM',           # Tasa de referencia de política monetaria (%)
    'cobre_lme': 'PN01652XM',          # Cobre LME (centavos US$/lb)
    'petroleo_wti': 'PN01660XM',       # Petróleo WTI (US$/barril)
    'pbi_indice': 'PN01770AM'          # Índice mensual de PBI (2007=100)
}

START_PERIOD = "2004-1"
END_PERIOD = "2026-5"

MESES_ES = {
    'Ene': 1, 'Feb': 2, 'Mar': 3, 'Abr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Ago': 8, 'Set': 9, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dic': 12
}

def parse_bcrp_period(period_str):
    parts = period_str.replace(' ', '').split('.')
    if len(parts) == 2:
        m_str, y_str = parts[0], parts[1]
        month = MESES_ES.get(m_str, 1)
        year = int(y_str)
        return pd.Period(year=year, month=month, freq='M')
    return None

print(f"Descargando series BCRP ({START_PERIOD} a {END_PERIOD})...")
series_frames = []

for var_name, code in BCRP_SERIES.items():
    url = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{code}/json/{START_PERIOD}/{END_PERIOD}/esp"
    try:
        resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=20)
        data = resp.json()
        periods = data.get('periods', [])
        
        idx = [parse_bcrp_period(p['name']) for p in periods]
        vals = []
        for p in periods:
            raw_val = p['values'][0]
            try:
                vals.append(float(raw_val) if raw_val not in ('n.d.', '', None) else np.nan)
            except ValueError:
                vals.append(np.nan)
        
        s = pd.Series(vals, index=pd.PeriodIndex(idx, freq='M'), name=var_name)
        series_frames.append(s)
        print(f"  [OK] {code} -> {var_name:20} ({len(s)} obs | Rango: {s.index.min()} a {s.index.max()})")
    except Exception as e:
        print(f"  [ERROR] {code} -> {var_name}: {e}")

df_bcrp = pd.concat(series_frames, axis=1).sort_index()
print(f"\nDataset BCRP: {df_bcrp.shape[0]} observaciones mensuales.")

### 2. Extracción del ICEN (IGP / ENFEN) y Mercados Globales (Yahoo Finance)

In [ ]:
# 2.1 Descarga del ICEN
print("Descargando ICEN desde met.igp.gob.pe...")
url_icen = "http://met.igp.gob.pe/datos/ICEN.txt"
resp_icen = requests.get(url_icen, headers={'User-Agent': 'Mozilla/5.0'}, timeout=20)

icen_rows = []
for line in resp_icen.text.splitlines():
    line = line.strip()
    if not line or line.startswith('%'):
        continue
    parts = line.split()
    if len(parts) >= 3:
        year = int(parts[0])
        month = int(parts[1])
        val = float(parts[2])
        icen_rows.append({
            'period': pd.Period(year=year, month=month, freq='M'),
            'icen': val
        })

df_icen = pd.DataFrame(icen_rows).set_index('period').loc[START_PERIOD:END_PERIOD]
print(f"ICEN descargado: {len(df_icen)} meses.")

# 2.2 Descarga de S&P 500 y VIX
print("Descargando S&P 500 (^GSPC) y VIX (^VIX) desde Yahoo Finance...")
yf_raw = yf.download(['^GSPC', '^VIX'], start='2004-01-01', end='2026-06-01', interval='1mo', progress=False)['Close']
yf_df = yf_raw.rename(columns={'^GSPC': 'sp500', '^VIX': 'vix'}).copy()
yf_df.index = pd.PeriodIndex(yf_df.index, freq='M')
yf_df = yf_df.loc[START_PERIOD:END_PERIOD]
print(f"Datos internacionales descargados: {len(yf_df)} meses.")

### 3. Transformaciones Econométricas y Ordenamiento Estricto de Columnas

Orden de variables:
$$\text{Fecha} \longrightarrow \text{BVL General (Nivel y Retorno)} \longrightarrow \text{ICEN (Shocks)} \longrightarrow \text{Expectativas} \longrightarrow \text{Controles Macro y Commodities} \longrightarrow \text{Controles Globales} \longrightarrow \text{Dummies}$$

In [ ]:
# Fusión base
df_raw = df_bcrp.join([df_icen, yf_df], how='outer').sort_index()

# Variables de tiempo
df_raw['date_str'] = df_raw.index.strftime('%Y-%m')
df_raw['year'] = df_raw.index.year
df_raw['month'] = df_raw.index.month
df_raw['stata_tm'] = (df_raw['year'] - 1960) * 12 + (df_raw['month'] - 1)

# Retornos Logarítmicos Continuos (r = 100 * ln(P_t / P_{t-1}))
price_vars = ['bvl_general', 'tc_pen_usd', 'cobre_lme', 'petroleo_wti', 'pbi_indice', 'sp500']
for col in price_vars:
    df_raw[f'ret_{col}'] = 100 * (np.log(df_raw[col]) - np.log(df_raw[col].shift(1)))

# Primeras diferencias en tasas y expectativas (puntos porcentuales)
diff_vars = ['tasa_ref', 'exp_inflacion_12m', 'exp_emp_economia_3m']
for var in diff_vars:
    df_raw[f'd_{var}'] = df_raw[var] - df_raw[var].shift(1)

# Shocks climáticos del ICEN
df_raw['icen_calido'] = df_raw['icen'].apply(lambda x: max(x, 0.0) if pd.notnull(x) else np.nan)
df_raw['icen_frio'] = df_raw['icen'].apply(lambda x: min(x, 0.0) if pd.notnull(x) else np.nan)
df_raw['dummy_nino'] = (df_raw['icen'] >= 1.0).astype(int)

# Dummies de quiebres exógenos
df_raw['dummy_crisis2008'] = ((df_raw.index >= '2008-09') & (df_raw.index <= '2009-06')).astype(int)
df_raw['dummy_covid'] = ((df_raw.index >= '2020-03') & (df_raw.index <= '2020-12')).astype(int)

# Ordenamiento estricto y lógico de columnas
ORDERED_COLUMNS = [
    # 1. Calendario y Fecha
    'date_str', 'year', 'month', 'stata_tm',
    
    # 2. Variable Dependiente Bursátil
    'bvl_general', 'ret_bvl_general',
    
    # 3. Variable Explicativa Climática Central
    'icen', 'icen_calido', 'icen_frio', 'dummy_nino',
    
    # 4. Canal de Expectativas (BCRP)
    'exp_inflacion_12m', 'd_exp_inflacion_12m',
    'exp_emp_economia_3m', 'd_exp_emp_economia_3m',
    
    # 5. Variables de Control Macroeconómicas y Commodities (BCRP)
    'tc_pen_usd', 'ret_tc_pen_usd',
    'tasa_ref', 'd_tasa_ref',
    'cobre_lme', 'ret_cobre_lme',
    'petroleo_wti', 'ret_petroleo_wti',
    'pbi_indice', 'ret_pbi_indice',
    
    # 6. Variables de Control Internacional (Yahoo Finance)
    'sp500', 'ret_sp500', 'vix',
    
    # 7. Controles de Quiebres Estructurales
    'dummy_crisis2008', 'dummy_covid'
]

df_clean = df_raw[ORDERED_COLUMNS].copy()

print("=== DATASET ORDENADO Y VALIDADO ===")
print(f"Dimensiones: {df_clean.shape[0]} meses × {df_clean.shape[1]} variables ordenadas")
print("Primeras 5 observaciones con columnas ordenadas:")
display(df_clean.head(5))
print("\nÚltimas 5 observaciones:")
display(df_clean.tail(5))

### 4. Estadísticas Descriptivas y Análisis Gráfico (2004 - 2026)

In [ ]:
cols_summary = ['ret_bvl_general', 'icen', 'exp_inflacion_12m', 'exp_emp_economia_3m', 'ret_tc_pen_usd', 'ret_cobre_lme', 'ret_sp500', 'vix']
summary_table = df_clean[cols_summary].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
summary_table.columns = ['N', 'Media', 'Desv. Est.', 'Mínimo', 'Mediana', 'Máximo']
display(summary_table.round(3))

# Gráficos de series temporales ordenadas
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), dpi=100, sharex=True)
dates = df_clean.index.to_timestamp()

# Panel A: ICEN vs Retorno BVL General
color_icen = '#d95f02'
ax1.set_title('A. Shocks Climáticos de El Niño Costero (ICEN) y Retornos de la BVL General (2004–2026)', fontsize=12, fontweight='bold')
ax1.set_ylabel('ICEN (°C)', color=color_icen, fontsize=10)
ax1.plot(dates, df_clean['icen'], color=color_icen, lw=2.0, label='ICEN (°C)')
ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax1.axhline(1.0, color='red', linestyle=':', alpha=0.7, label='Umbral El Niño (+1.0°C)')
ax1.axhline(-1.0, color='blue', linestyle=':', alpha=0.7, label='Umbral La Niña (-1.0°C)')
ax1.tick_params(axis='y', labelcolor=color_icen)
ax1.legend(loc='upper left')

ax1_b = ax1.twinx()
color_bvl = '#1b9e77'
ax1_b.set_ylabel('Retorno Mensual BVL (%)', color=color_bvl, fontsize=10)
ax1_b.plot(dates, df_clean['ret_bvl_general'], color=color_bvl, alpha=0.45, lw=1.2, label='Retorno BVL General (%)')
ax1_b.tick_params(axis='y', labelcolor=color_bvl)

# Panel B: Canal de Expectativas
ax2.set_title('B. Canal Mediador: Expectativa de Inflación a 12m y Confianza Empresarial de la Economía a 3m', fontsize=12, fontweight='bold')
ax2.set_xlabel('Año', fontsize=10)
ax2.set_ylabel('Expectativa Inflación 12m (%)', color='#2b5c8f', fontsize=10)
ax2.plot(dates, df_clean['exp_inflacion_12m'], color='#2b5c8f', lw=1.8, label='Expectativa Inflación 12m (%)')
ax2.axhline(2.0, color='gray', linestyle=':', alpha=0.4, label='Rango Meta BCRP (1-3%)')
ax2.axhline(3.0, color='gray', linestyle=':', alpha=0.4)
ax2.tick_params(axis='y', labelcolor='#2b5c8f')
ax2.legend(loc='upper left')

ax2_b = ax2.twinx()
color_conf = '#7570b3'
ax2_b.set_ylabel('Confianza Economía 3m (Puntos)', color=color_conf, fontsize=10)
ax2_b.plot(dates, df_clean['exp_emp_economia_3m'], color=color_conf, lw=1.8, linestyle='--', label='Confianza Economía 3m')
ax2_b.axhline(50, color='red', linestyle=':', alpha=0.6, label='Umbral Neutro (50)')
ax2_b.tick_params(axis='y', labelcolor=color_conf)
ax2_b.legend(loc='upper right')

fig.tight_layout()
plt.show()

### 5. Exportación en Orden Estricto (CSV y Stata .dta)

In [ ]:
# Exportar datasets en orden estricto de columnas
files_to_export = ['data_bvl_icen.csv', 'data_bvl_icen.csv']
for f_name in files_to_export:
    df_clean.to_csv(f_name, index=False)
    print(f"Exportado CSV ordenado: {f_name} ({df_clean.shape[0]} filas × {df_clean.shape[1]} columnas)")

dta_files = ['data_bvl_icen.dta', 'data_bvl_icen.dta']
for d_name in dta_files:
    try:
        df_clean.to_stata(d_name, write_index=False, version=118)
        print(f"Exportado Stata ordenado: {d_name}")
    except Exception as e:
        print(f"Aviso al exportar a {d_name}: {e}")

print("\n¡Datasets perfectamente sincronizados y ordenados!")